In [1]:
!pip -q install --no-cache-dir "numpy==1.26.4" "pandas==2.2.2"
!pip -q install --no-cache-dir "opencv-python-headless==4.10.0.84"
!pip -q install --no-cache-dir decord tqdm einops
!pip -q install --no-cache-dir torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install --no-cache-dir "ultralytics==8.3.0"
!pip -q install --no-cache-dir "emotiefflib[torch]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 159.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 53.1 MB/s eta 0:00:00


In [2]:
import os
from pathlib import Path

ROOT = "/content/miga_data"
SRC_DIR = "/content/src"

Path(ROOT).mkdir(parents=True, exist_ok=True)
Path(SRC_DIR).mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("SRC_DIR:", SRC_DIR)


ROOT: /content/miga_data
SRC_DIR: /content/src


In [3]:
from google.colab import drive
drive.mount("/content/drive")


FEAT_DIR = "/content/drive/MyDrive/miga_features_cache_agcn"


Mounted at /content/drive


In [4]:
DATA_DIR = "/content/miga_data"
!mkdir -p $DATA_DIR

!wget -O $DATA_DIR/imigue_skeleton_phase1.zip https://miga3.a3s.fi/imigue_skeleton_phase1.zip

!wget -O $DATA_DIR/imigue_rgb_phase1.zip https://miga3.a3s.fi/imigue_rgb_phase1.zip

!wget -O $DATA_DIR/imigue_skeleton_phase2.zip https://miga3.a3s.fi/imigue_skeleton_phase2.zip

!wget -O $DATA_DIR/imigue_rgb_phase2.zip https://miga3.a3s.fi/imigue_rgb_phase2.zip

--2026-02-17 06:37:38--  https://miga3.a3s.fi/imigue_skeleton_phase1.zip
Resolving miga3.a3s.fi (miga3.a3s.fi)... 86.50.254.18, 86.50.254.19
Connecting to miga3.a3s.fi (miga3.a3s.fi)|86.50.254.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6270753807 (5.8G) [application/zip]
Saving to: ‘/content/miga_data/imigue_skeleton_phase1.zip’

/content/miga_data/ 100%[===================>]   5.84G  15.2MB/s    in 5m 22s  

2026-02-17 06:43:02 (18.6 MB/s) - ‘/content/miga_data/imigue_skeleton_phase1.zip’ saved [6270753807/6270753807]

--2026-02-17 06:43:02--  https://miga3.a3s.fi/imigue_rgb_phase1.zip
Resolving miga3.a3s.fi (miga3.a3s.fi)... 86.50.254.18, 86.50.254.19
Connecting to miga3.a3s.fi (miga3.a3s.fi)|86.50.254.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6512356945 (6.1G) [application/zip]
Saving to: ‘/content/miga_data/imigue_rgb_phase1.zip’

/content/miga_data/ 100%[===================>]   6.06G  20.4MB/s    in 5m 52s  

202

In [5]:
!unzip -q $DATA_DIR/imigue_skeleton_phase1.zip -d $DATA_DIR
!unzip -q $DATA_DIR/imigue_rgb_phase1.zip -d $DATA_DIR
!unzip -q $DATA_DIR/imigue_skeleton_phase2.zip -d $DATA_DIR
!unzip -q $DATA_DIR/imigue_rgb_phase2.zip -d $DATA_DIR

In [7]:
!rm $DATA_DIR/*.zip

In [6]:
%%writefile /content/src/__init__.py

Writing /content/src/__init__.py


In [8]:
%%writefile /content/src/paths.py
import os, glob

ROOT = "/content/miga_data"

RGB_P1_TRAIN = f"{ROOT}/imigue_rgb_phase1/train_data"
RGB_P1_VAL   = f"{ROOT}/imigue_rgb_phase1/validation_data"
RGB_P2       = f"{ROOT}/imigue_rgb_phase2"

SK_P1_TRAIN  = f"{ROOT}/imigue_data_phase1/datasets/imigue_skeleton_train"
SK_P1_VAL    = f"{ROOT}/imigue_data_phase1/datasets/imigue_skeleton_validate"
SK_P2_TEST   = f"{ROOT}/imigue_data_phase2/imigue_skeleton_test"

def vid4(x): return f"{int(x):04d}"

def resolve_video_path_phase1(video_id, split):
    v = vid4(video_id)
    if split == "train":
        p = os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    if split == "val":
        p = os.path.join(RGB_P1_VAL, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    for p in [
        os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4"),
        os.path.join(RGB_P1_VAL, v, f"{v}.mp4"),
    ]:
        if os.path.exists(p): return p
    return None

def resolve_video_path_phase2(video_id):
    v = vid4(video_id)
    p = os.path.join(RGB_P2, v, f"{v}.mp4")
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(RGB_P2, "**", f"{v}.mp4"), recursive=True)
    return hits[0] if hits else None

def resolve_skeleton_path_phase1(video_id, split, prefer_hand=True):
    v = vid4(video_id)
    base = SK_P1_TRAIN if split=="train" else SK_P1_VAL
    p_hand  = os.path.join(base, v, f"{v}_light_hand.csv")
    p_light = os.path.join(base, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None

def resolve_skeleton_path_phase2(video_id, prefer_hand=True):
    v = vid4(video_id)
    p_hand  = os.path.join(SK_P2_TEST, v, f"{v}_light_hand.csv")
    p_light = os.path.join(SK_P2_TEST, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None


Writing /content/src/paths.py


In [9]:
import os
import pandas as pd
import glob, os

ROOT = "/content/miga_data"

train_csv = f"{ROOT}/imigue_rgb_phase1/train_label.csv"
val_csv   = f"{ROOT}/imigue_rgb_phase1/validation_label.csv"

train_df = pd.read_csv(train_csv)
val_df   = pd.read_csv(val_csv)

train_df["split"] = "train"
val_df["split"] = "val"

phase1_all = pd.concat([train_df, val_df], ignore_index=True)

def vid4(x):
    return f"{int(x):04d}"

def video_path(row):
    v = vid4(row["video_id"])
    if row["split"] == "train":
        return f"{ROOT}/imigue_rgb_phase1/train_data/{v}/{v}.mp4"
    else:
        return f"{ROOT}/imigue_rgb_phase1/validation_data/{v}/{v}.mp4"

def skeleton_path(row):
    v = f"{int(row['video_id']):04d}"
    roots = [
        "/content/miga_data/imigue_data_phase1",
        "/content/miga_data",
    ]
    for root in roots:
        hits = glob.glob(f"{root}/**/{v}*_light*.csv", recursive=True)
        if hits:
            return hits[0]
    return None


phase1_all["skeleton_path"] = phase1_all.apply(skeleton_path, axis=1)
phase1_all.to_csv("/content/phase1_all_with_split.csv", index=False)


In [10]:
%%writefile /content/src/agcn.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class AGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_t = nn.Conv2d(in_channels, out_channels, kernel_size=(9, 1), padding=(4, 0))
        self.conv_v = nn.Conv2d(out_channels, out_channels, kernel_size=(1, 1))
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv_t(x)
        x = self.conv_v(x)
        x = self.bn(x)
        return F.relu(x)


class AGCNModel(nn.Module):

    def __init__(self, in_channels=3, hidden=64, out_dim=512):
        super().__init__()

        self.data_bn = nn.BatchNorm1d(in_channels * 25)

        self.block1 = AGCNBlock(in_channels, hidden)
        self.block2 = AGCNBlock(hidden, hidden * 2)
        self.block3 = AGCNBlock(hidden * 2, hidden * 4)

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(hidden * 4, out_dim)

    def forward(self, x):
        B, T, V, C = x.shape

        x = x.permute(0, 3, 1, 2).contiguous()

        x = x.view(B, C * V, T)
        x = self.data_bn(x)
        x = x.view(B, C, V, T).permute(0, 1, 3, 2)

        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.pool(x).view(B, -1)
        x = self.fc(x)
        return x


Writing /content/src/agcn.py


In [11]:
FEAT_DRIVE = "/content/drive/MyDrive/miga_features_cache_agcn"
CSV = "/content/phase1_all_with_split.csv"
PHASE = 1

print("BASE FEAT_DRIVE:", FEAT_DRIVE)

BASE FEAT_DRIVE: /content/drive/MyDrive/miga_features_cache_agcn


In [12]:
%%writefile /content/src/feature_extract.py
import os
import glob
import hashlib
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from decord import VideoReader, cpu


MIGA_ROOT = os.environ.get("MIGA_ROOT", "/content/miga_data")

RGB_P1_TRAIN = f"{MIGA_ROOT}/imigue_rgb_phase1/train_data"
RGB_P1_VAL   = f"{MIGA_ROOT}/imigue_rgb_phase1/validation_data"
RGB_P2       = f"{MIGA_ROOT}/imigue_rgb_phase2"

SK_P1_TRAIN  = f"{MIGA_ROOT}/imigue_data_phase1/datasets/imigue_skeleton_train"
SK_P1_VAL    = f"{MIGA_ROOT}/imigue_data_phase1/datasets/imigue_skeleton_validate"
SK_P2_TEST   = f"{MIGA_ROOT}/imigue_data_phase2/imigue_skeleton_test"

# ---------------------------
# EXP SWITCHES (defaults = baseline)
# ---------------------------
CTX_BACKBONE  = os.environ.get("CTX_BACKBONE", "r3d_18")     # r3d_18 | mc3_18 | r2plus1d_18
FACE_SAMPLING = os.environ.get("FACE_SAMPLING", "mid")      # mid | 3frames_avg
FACE_DETECT   = os.environ.get("FACE_DETECT", "yolo")       # yolo | center
SKEL_AGG      = os.environ.get("SKEL_AGG", "mean")          # mean | mean_std | max


def vid4(x: int) -> str:
    return f"{int(x):04d}"


def resolve_video_path_phase1(video_id: int, split: str) -> Optional[str]:
    v = vid4(video_id)
    if split == "train":
        p = os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    if split == "val":
        p = os.path.join(RGB_P1_VAL, v, f"{v}.mp4")
        return p if os.path.exists(p) else None
    for p in [
        os.path.join(RGB_P1_TRAIN, v, f"{v}.mp4"),
        os.path.join(RGB_P1_VAL, v, f"{v}.mp4"),
    ]:
        if os.path.exists(p):
            return p
    return None


def resolve_video_path_phase2(video_id: int) -> Optional[str]:
    v = vid4(video_id)
    p = os.path.join(RGB_P2, v, f"{v}.mp4")
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(RGB_P2, "**", f"{v}.mp4"), recursive=True)
    return hits[0] if hits else None


def resolve_skeleton_path_phase1(video_id: int, split: str, prefer_hand: bool = True) -> Optional[str]:
    v = vid4(video_id)
    base = SK_P1_TRAIN if split == "train" else SK_P1_VAL
    p_hand  = os.path.join(base, v, f"{v}_light_hand.csv")
    p_light = os.path.join(base, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None


def resolve_skeleton_path_phase2(video_id: int, prefer_hand: bool = True) -> Optional[str]:
    v = vid4(video_id)
    p_hand  = os.path.join(SK_P2_TEST, v, f"{v}_light_hand.csv")
    p_light = os.path.join(SK_P2_TEST, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand): return p_hand
    if os.path.exists(p_light): return p_light
    if os.path.exists(p_hand):  return p_hand
    return None


def cache_key(video_id: int, split: str, phase: int) -> str:
    s = f"{int(video_id)}|{split}|{phase}"
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]


def cache_paths(feat_dir: str, video_id: int, split: str, phase: int) -> Dict[str, str]:
    key = cache_key(video_id, split, phase)
    d = Path(feat_dir)
    d.mkdir(parents=True, exist_ok=True)
    return {
        "ctx":   str(d / f"{key}_ctx.pt"),
        "face":  str(d / f"{key}_face.pt"),
        "fmask": str(d / f"{key}_facemask.pt"),
        "skel":  str(d / f"{key}_skel.pt"),
        "smask": str(d / f"{key}_skelmask.pt"),
    }


def _vr(path: str) -> VideoReader:
    return VideoReader(path, ctx=cpu(0))


# ---------------------------
# CTX
# ---------------------------
_CTX_MODEL = None

def build_ctx_model(device: str = "cuda"):
    global _CTX_MODEL
    if _CTX_MODEL is not None:
        _CTX_MODEL.to(device).eval()
        return _CTX_MODEL

    if CTX_BACKBONE == "mc3_18":
        from torchvision.models.video import mc3_18, MC3_18_Weights
        weights = MC3_18_Weights.DEFAULT
        model = mc3_18(weights=weights)
        model.fc = nn.Identity()
    elif CTX_BACKBONE == "r2plus1d_18":
        from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
        weights = R2Plus1D_18_Weights.DEFAULT
        model = r2plus1d_18(weights=weights)
        model.fc = nn.Identity()
    else:
        from torchvision.models.video import r3d_18, R3D_18_Weights
        weights = R3D_18_Weights.DEFAULT
        model = r3d_18(weights=weights)
        model.fc = nn.Identity()

    model = model.to(device).eval()
    _CTX_MODEL = model
    return _CTX_MODEL


def _preprocess_clip_torch(clip_rgb: np.ndarray, device: str) -> torch.Tensor:
    x = torch.from_numpy(clip_rgb).to(torch.float32) / 255.0
    x = x.permute(3, 0, 1, 2)  # (C,T,H,W)
    x = x.unsqueeze(0)         # (1,C,T,H,W)
    x = F.interpolate(x, size=(x.shape[2], 112, 112), mode="trilinear", align_corners=False)

    mean = torch.tensor([0.43216, 0.394666, 0.37645], device=x.device).view(1,3,1,1,1)
    std  = torch.tensor([0.22803, 0.22145, 0.216989], device=x.device).view(1,3,1,1,1)
    x = (x - mean) / std
    return x.to(device)


@torch.no_grad()
def extract_ctx_stream(video_path: str, device: str = "cuda", chunk: int = 32) -> torch.Tensor:
    model = build_ctx_model(device=device)
    vr = _vr(video_path)
    T = len(vr)
    if T <= 0:
        return torch.zeros((0, 512), dtype=torch.float32)

    pad = (-T) % chunk
    total = T + pad

    feats = []
    for i in range(0, total, chunk):
        idxs = list(range(i, min(i + chunk, T)))
        if len(idxs) == 0:
            break
        frames = vr.get_batch(idxs).asnumpy()
        if frames.shape[0] < chunk:
            frames = np.concatenate([frames, np.repeat(frames[-1:], chunk - frames.shape[0], axis=0)], axis=0)

        x = _preprocess_clip_torch(frames, device=device)
        f = model(x).squeeze(0).detach().cpu()
        feats.append(f)

    return torch.stack(feats, dim=0).to(torch.float32)


# ---------------------------
# FACE
# ---------------------------
_FACE_MODEL = None
_EMO_REC = None

def _init_yolo():
    global _FACE_MODEL
    if _FACE_MODEL is not None:
        return _FACE_MODEL
    try:
        from ultralytics import YOLO
        for w in ["yolov8n-face.pt", "yolov8n.pt"]:
            try:
                _FACE_MODEL = YOLO(w)
                break
            except Exception:
                _FACE_MODEL = None
    except Exception:
        _FACE_MODEL = None
    return _FACE_MODEL


def _center_crop(img, size=224):
    h, w = img.shape[:2]
    s = min(h, w)
    y0 = (h - s) // 2
    x0 = (w - s) // 2
    crop = img[y0:y0+s, x0:x0+s]
    import cv2
    crop = cv2.resize(crop, (size, size))
    return crop


def _detect_face_crop(img_rgb, size=224):
    if FACE_DETECT == "center":
        return _center_crop(img_rgb, size=size), True

    model = _init_yolo()
    if model is None:
        return _center_crop(img_rgb, size=size), False

    try:
        res = model.predict(source=img_rgb, verbose=False)
        if len(res) == 0 or res[0].boxes is None or len(res[0].boxes) == 0:
            return _center_crop(img_rgb, size=size), False
        boxes = res[0].boxes.xyxy.detach().cpu().numpy()
        areas = (boxes[:,2]-boxes[:,0])*(boxes[:,3]-boxes[:,1])
        b = boxes[int(np.argmax(areas))]
        x1, y1, x2, y2 = [int(max(0, v)) for v in b]
        x2 = min(x2, img_rgb.shape[1]-1)
        y2 = min(y2, img_rgb.shape[0]-1)
        crop = img_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            return _center_crop(img_rgb, size=size), False
        import cv2
        crop = cv2.resize(crop, (size, size))
        return crop, True
    except Exception:
        return _center_crop(img_rgb, size=size), False


def _init_emotieff(device: str):
    global _EMO_REC
    if _EMO_REC is not None:
        return _EMO_REC
    from emotiefflib.facial_analysis import EmotiEffLibRecognizerTorch
    try:
        _EMO_REC = EmotiEffLibRecognizerTorch(device=device)
    except TypeError:
        _EMO_REC = EmotiEffLibRecognizerTorch()
    return _EMO_REC


def _as_rgb_uint8(face_rgb_224):
    x = np.asarray(face_rgb_224)
    if x.dtype != np.uint8:
        x = np.clip(x, 0, 255).astype(np.uint8)
    return x


@torch.no_grad()
def emotieff_embed(face_rgb_224: np.ndarray, device: str = "cuda") -> np.ndarray:
    rec = _init_emotieff(device=device)
    x = _as_rgb_uint8(face_rgb_224)

    for name in ["get_face_embedding","get_embedding","get_embeddings","extract_embedding","extract_embeddings","get_features","extract_features"]:
        if hasattr(rec, name) and callable(getattr(rec, name)):
            out = getattr(rec, name)(x)
            return np.asarray(out).reshape(-1).astype(np.float32)

    if callable(rec):
        out = rec(x)
        if isinstance(out, dict):
            for k in ["embedding","emb","features","feature"]:
                if k in out:
                    return np.asarray(out[k]).reshape(-1).astype(np.float32)
        return np.asarray(out).reshape(-1).astype(np.float32)

    raise RuntimeError("EmotiEffLib: не удалось получить embedding")


@torch.no_grad()
def extract_face_windows(video_path: str, device: str = "cuda", chunk: int = 32) -> Tuple[torch.Tensor, torch.Tensor]:
    vr = _vr(video_path)
    T = len(vr)
    if T <= 0:
        return torch.zeros((0, 1280), dtype=torch.float32), torch.zeros((0,), dtype=torch.bool)

    pad = (-T) % chunk
    total = T + pad

    feats = []
    mask = []
    D = None

    for i in range(0, total, chunk):
        if FACE_SAMPLING == "3frames_avg":
            idxs = [
                min(i + max(0, chunk // 4), T - 1),
                min(i + max(0, chunk // 2), T - 1),
                min(i + max(0, (3 * chunk) // 4), T - 1),
            ]
        else:
            idxs = [min(i + chunk // 2, T - 1)]

        embs = []
        oks = []
        for idx in idxs:
            frame = vr[idx].asnumpy()
            crop, ok = _detect_face_crop(frame, size=224)
            if ok:
                embs.append(emotieff_embed(crop, device=device))
                oks.append(True)
            else:
                oks.append(False)

        if any(oks):
            emb = np.mean(embs, axis=0).astype(np.float32)
            if D is None:
                D = int(emb.shape[0])
            feats.append(torch.from_numpy(emb).to(torch.float32))
            mask.append(True)
        else:
            if D is None:
                D = 1280
            feats.append(torch.zeros((D,), dtype=torch.float32))
            mask.append(False)

    face = torch.stack(feats, dim=0)
    fmask = torch.tensor(mask, dtype=torch.bool)
    return face, fmask


# ---------------------------
# SKEL
# ---------------------------
_SKEL_PROJ = None

def _skel_projector(in_dim: int, out_dim: int = 512) -> nn.Module:
    global _SKEL_PROJ
    if _SKEL_PROJ is not None and getattr(_SKEL_PROJ, "_in_dim", None) == in_dim:
        return _SKEL_PROJ
    torch.manual_seed(42)
    proj = nn.Sequential(
        nn.Linear(in_dim, out_dim),
        nn.LayerNorm(out_dim),
        nn.GELU(),
    )
    proj._in_dim = in_dim
    _SKEL_PROJ = proj.eval()
    return _SKEL_PROJ


def _load_skeleton_csv(path: str) -> np.ndarray:
    df = pd.read_csv(path)
    return df.values.astype(np.float32)  # (T,D)


@torch.no_grad()
def extract_skeleton_windows(skel_csv_path: str, device: str = "cpu", chunk: int = 32) -> Tuple[torch.Tensor, torch.Tensor]:
    arr = _load_skeleton_csv(skel_csv_path)
    T, D = arr.shape
    if T <= 0:
        return torch.zeros((0, 512), dtype=torch.float32), torch.zeros((0,), dtype=torch.bool)

    pad = (-T) % chunk
    total = T + pad
    if pad:
        arr = np.concatenate([arr, np.repeat(arr[-1:], pad, axis=0)], axis=0)

    # aggregator output dim
    if SKEL_AGG == "mean_std":
        in_dim = 2 * D
    else:
        in_dim = D

    proj = _skel_projector(in_dim, 512)

    feats = []
    mask = []
    for i in range(0, total, chunk):
        w = arr[i:i+chunk]  # (chunk,D)
        valid = not np.allclose(w, 0)

        if SKEL_AGG == "max":
            v = w.max(axis=0)
        elif SKEL_AGG == "mean_std":
            mu = w.mean(axis=0)
            sd = w.std(axis=0)
            v = np.concatenate([mu, sd], axis=0)
        else:
            v = w.mean(axis=0)

        x = torch.from_numpy(v).to(torch.float32)
        z = proj(x).detach().cpu()
        feats.append(z)
        mask.append(valid)

    skel = torch.stack(feats, dim=0).to(torch.float32)
    smask = torch.tensor(mask, dtype=torch.bool)
    return skel, smask


@torch.no_grad()
def build_and_cache_one(
    video_id: int,
    split: str,
    phase: int,
    feat_dir: str,
    chunk: int = 32,
    vpath: Optional[str] = None,
    spath: Optional[str] = None,
    do_ctx: bool = True,
    do_face: bool = True,
    do_skel: bool = True,
    device: str = "cuda",
):
    ps = cache_paths(feat_dir, video_id, split, phase)

    if vpath is None:
        vpath = resolve_video_path_phase1(video_id, split) if phase == 1 else resolve_video_path_phase2(video_id)

    if spath is None:
        spath = resolve_skeleton_path_phase1(video_id, split) if phase == 1 else resolve_skeleton_path_phase2(video_id)

    if do_ctx or do_face:
        if vpath is None or (not os.path.exists(vpath)):
            raise FileNotFoundError(f"RGB video not found for id={video_id} split={split} phase={phase}")

    if do_skel:
        if spath is None or (not os.path.exists(spath)):
            raise FileNotFoundError(f"Skeleton csv not found for id={video_id} split={split} phase={phase}")

    if do_ctx and (not os.path.exists(ps["ctx"])):
        ctx = extract_ctx_stream(vpath, device=device, chunk=chunk)
        torch.save(ctx, ps["ctx"])

    if do_face and (not os.path.exists(ps["face"])) and (not os.path.exists(ps["fmask"])):
        face, fmask = extract_face_windows(vpath, device=device, chunk=chunk)
        torch.save(face, ps["face"])
        torch.save(fmask, ps["fmask"])

    if do_skel and (not os.path.exists(ps["skel"])) and (not os.path.exists(ps["smask"])):
        skel, smask = extract_skeleton_windows(spath, device="cpu", chunk=chunk)
        torch.save(skel, ps["skel"])
        torch.save(smask, ps["smask"])

    return ps


Writing /content/src/feature_extract.py


In [13]:
%%writefile /content/src/cache_features_cli.py
import argparse
import pandas as pd
from tqdm import tqdm
import torch
import traceback

from src.feature_extract import build_and_cache_one

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True)
    ap.add_argument("--phase", type=int, required=True)
    ap.add_argument("--feat_dir", required=True)
    ap.add_argument("--chunk", type=int, default=32)
    ap.add_argument("--start", type=int, default=0)
    ap.add_argument("--end", type=int, default=999999)
    ap.add_argument("--split_col", type=str, default="split")
    ap.add_argument("--device", choices=["auto","cpu","cuda"], default="auto")
    ap.add_argument("--do_ctx", action="store_true")
    ap.add_argument("--do_face", action="store_true")
    ap.add_argument("--do_skel", action="store_true")
    args = ap.parse_args()

    if args.device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        device = args.device

    if not (args.do_ctx or args.do_face or args.do_skel):
        args.do_ctx = args.do_face = args.do_skel = True

    print("DEVICE:", device)
    print("FEAT_DIR:", args.feat_dir)
    print("MODES: ctx=", args.do_ctx, "face=", args.do_face, "skel=", args.do_skel)

    df = pd.read_csv(args.csv)
    df = df.iloc[args.start:args.end].reset_index(drop=True)

    ok, bad = 0, 0

    for i, r in tqdm(df.iterrows(), total=len(df)):
        try:
            vid = int(r["video_id"])
            split = r.get(args.split_col, "test")

            build_and_cache_one(
                video_id=vid,
                split=split,
                phase=args.phase,
                feat_dir=args.feat_dir,
                chunk=args.chunk,
                vpath=None,
                spath=r.get("skeleton_path", None),
                do_ctx=args.do_ctx,
                do_face=args.do_face,
                do_skel=args.do_skel,
                device=device,
            )
            ok += 1

        except Exception:
            bad += 1
            print(" ERROR")
            print("row:", i, "video_id:", r.get("video_id"), "split:", r.get(args.split_col, ""))
            if "skeleton_path" in r:
                print("skeleton_path:", r.get("skeleton_path"))
            traceback.print_exc()

    print(f"\nDONE shard [{args.start}:{args.end}) ok={ok} bad={bad}")

if __name__ == "__main__":
    main()


Writing /content/src/cache_features_cli.py


**Исследование вариантов создания эмбеддингов**

**CTX Variant 0: r3d_18**

In [ ]:
%%bash
export CTX_BACKBONE=r3d_18

CHUNK=64
SHARD=10
DEVICE=cuda
CSV=/content/phase1_all_with_split.csv
BASE_FEAT_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_NAME=baseline_ctx
FEAT_DIR=${BASE_FEAT_DIR}/${EXP_NAME}

mkdir -p "$FEAT_DIR"

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== CTX baseline (r3d_18) shard $START:$END (chunk=$CHUNK) ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 1 --feat_dir "$FEAT_DIR" \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_ctx

  START=$END
done


=== CTX baseline (r3d_18) shard 0:10 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_ctx
MODES: ctx= True face= False skel= False
Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth

DONE shard [0:10) ok=10 bad=0
=== CTX baseline (r3d_18) shard 10:20 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_ctx
MODES: ctx= True face= False skel= False

DONE shard [10:20) ok=10 bad=0
=== CTX baseline (r3d_18) shard 20:30 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_ctx
MODES: ctx= True face= False skel= False

DONE shard [20:30) ok=10 bad=0
=== CTX baseline (r3d_18) shard 30:40 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_ctx
MODES: ctx= True face= False skel= False

DONE shard [30:40) ok=10 bad=0
=== CTX baseline (r3d_18) shard 4

  0%|          | 0/10 [00:00<?, ?it/s]
  0%|          | 0.00/127M [00:00<?, ?B/s]
  2%|▏         | 2.25M/127M [00:00<00:05, 23.2MB/s]
  4%|▎         | 4.50M/127M [00:00<00:05, 22.7MB/s]
 16%|█▋        | 20.9M/127M [00:00<00:01, 89.1MB/s]
 30%|███       | 38.2M/127M [00:00<00:00, 125MB/s] 
 47%|████▋     | 60.4M/127M [00:00<00:00, 163MB/s]
 60%|█████▉    | 76.1M/127M [00:00<00:00, 156MB/s]
 72%|███████▏  | 91.1M/127M [00:00<00:00, 149MB/s]
100%|██████████| 127M/127M [00:00<00:00, 147MB/s]
100%|██████████| 5/5 [04:47<00:00, 57.54s/it]


**CTX Variant 1: exp_ctx_m**

In [ ]:
%%bash
set -euxo pipefail
CSV=/content/phase1_all_with_split.csv
BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3

# 1) symlink all baseline into exp (fast)
mkdir -p "$EXP_DIR"
ln -sfn "$BASE_DIR"/*_face.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_facemask.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skel.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skelmask.pt "$EXP_DIR"/ 2>/dev/null || true

# 2) recompute ONLY ctx into exp
DEVICE=cuda
CHUNK=32
SHARD=10
export PYTHONUNBUFFERED=1

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD)); if [ $END -gt $TOTAL ]; then END=$TOTAL; fi
  echo "=== CTX mc3_18 shard $START:$END ==="
  time env CTX_BACKBONE=mc3_18 PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 \
    python -u /content/src/cache_features_cli.py \
      --csv "$CSV" --phase 1 --feat_dir "$EXP_DIR" \
      --chunk "$CHUNK" --start "$START" --end "$END" --device "$DEVICE" \
      --do_ctx
  START=$END
done


=== CTX mc3_18 shard 0:10 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3
MODES: ctx= True face= False skel= False

DONE shard [0:10) ok=10 bad=0
=== CTX mc3_18 shard 10:20 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3
MODES: ctx= True face= False skel= False

DONE shard [10:20) ok=10 bad=0
=== CTX mc3_18 shard 20:30 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3
MODES: ctx= True face= False skel= False

DONE shard [20:30) ok=10 bad=0
=== CTX mc3_18 shard 30:40 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3
MODES: ctx= True face= False skel= False

DONE shard [30:40) ok=10 bad=0
=== CTX mc3_18 shard 40:50 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3
MODES: ctx= True face= False skel= False

DONE shard [40:50) ok=10 bad=0
=== CTX mc3_18 shard 50:60 ===
DEVICE: cuda
FEAT_DIR: /content/driv

+ CSV=/content/phase1_all_with_split.csv
+ BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
+ EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3
+ mkdir -p /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_mc3
+ ln -sfn /content/drive/MyDrive/miga_features_cache_agcn/00a3ed33d38e_face.pt /content/drive/MyDrive/miga_features_cache_agcn/00b40a2e4d97_face.pt /content/drive/MyDrive/miga_features_cache_agcn/01087f1b2489_face.pt /content/drive/MyDrive/miga_features_cache_agcn/0123fc67d163_face.pt /content/drive/MyDrive/miga_features_cache_agcn/017fc7925072_face.pt /content/drive/MyDrive/miga_features_cache_agcn/01b372587e0a_face.pt /content/drive/MyDrive/miga_features_cache_agcn/01dc5000e304_face.pt /content/drive/MyDrive/miga_features_cache_agcn/03ae04661248_face.pt /content/drive/MyDrive/miga_features_cache_agcn/049e754be9f7_face.pt /content/drive/MyDrive/miga_features_cache_agcn/04c0dc00860f_face.pt /content/drive/MyDrive/miga_features_cache_agcn/05b600b0301

**CTX Variant 2: exp_ctx_r2plus1d**

In [ ]:
%%bash
set -euxo pipefail
CSV=/content/phase1_all_with_split.csv
BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d

mkdir -p "$EXP_DIR"
ln -sfn "$BASE_DIR"/*_face.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_facemask.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skel.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skelmask.pt "$EXP_DIR"/ 2>/dev/null || true

DEVICE=cuda
CHUNK=32
SHARD=10
export PYTHONUNBUFFERED=1

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD)); if [ $END -gt $TOTAL ]; then END=$TOTAL; fi
  echo "=== CTX r2plus1d_18 shard $START:$END ==="
  time env CTX_BACKBONE=r2plus1d_18 PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 \
    python -u /content/src/cache_features_cli.py \
      --csv "$CSV" --phase 1 --feat_dir "$EXP_DIR" \
      --chunk "$CHUNK" --start "$START" --end "$END" --device "$DEVICE" \
      --do_ctx
  START=$END
done


=== CTX r2plus1d_18 shard 0:10 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False
Downloading: "https://download.pytorch.org/models/r2plus1d_18-91a641e6.pth" to /root/.cache/torch/hub/checkpoints/r2plus1d_18-91a641e6.pth

DONE shard [0:10) ok=10 bad=0
=== CTX r2plus1d_18 shard 10:20 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False

DONE shard [10:20) ok=10 bad=0
=== CTX r2plus1d_18 shard 20:30 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False

DONE shard [20:30) ok=10 bad=0
=== CTX r2plus1d_18 shard 30:40 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False

DONE shard [30:40) ok=10 bad=0
=== CTX r2plus1d_18 shard 40:50 ===
DEVICE: cuda
FEAT_DIR: /content/drive/M

+ CSV=/content/phase1_all_with_split.csv
+ BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
+ EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d
+ mkdir -p /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d
+ ln -sfn /content/drive/MyDrive/miga_features_cache_agcn/00a3ed33d38e_face.pt /content/drive/MyDrive/miga_features_cache_agcn/00b40a2e4d97_face.pt /content/drive/MyDrive/miga_features_cache_agcn/01087f1b2489_face.pt /content/drive/MyDrive/miga_features_cache_agcn/0123fc67d163_face.pt /content/drive/MyDrive/miga_features_cache_agcn/017fc7925072_face.pt /content/drive/MyDrive/miga_features_cache_agcn/01b372587e0a_face.pt /content/drive/MyDrive/miga_features_cache_agcn/01dc5000e304_face.pt /content/drive/MyDrive/miga_features_cache_agcn/03ae04661248_face.pt /content/drive/MyDrive/miga_features_cache_agcn/049e754be9f7_face.pt /content/drive/MyDrive/miga_features_cache_agcn/04c0dc00860f_face.pt /content/drive/MyDrive/miga_features_cache_agcn/0

**FACE Variant 0: yolo+mid**

In [14]:
%%bash
export FACE_SAMPLING=mid
export FACE_DETECT=yolo

CHUNK=32
SHARD=5
DEVICE=cuda
CSV=/content/phase1_all_with_split.csv
BASE_FEAT_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_NAME=baseline_face
FEAT_DIR=${BASE_FEAT_DIR}/${EXP_NAME}

mkdir -p "$FEAT_DIR"

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== FACE baseline (mid + yolo) shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 1 --feat_dir "$FEAT_DIR" \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_face

  START=$END
done

=== FACE baseline (mid + yolo) shard 0:5 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_face
MODES: ctx= False face= True skel= False
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

DONE shard [0:5) ok=5 bad=0
=== FACE baseline (mid + yolo) shard 5:10 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_face
MODES: ctx= False face= True skel= False

DONE shard [5:10) ok=5 bad=0
=== FACE baseline (mid + yolo) shard 10:15 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_face
MODES: ctx= False face= True skel= False

DONE shard [10:15) ok=5 bad=0
=== FACE baseline (mid + yolo) shard 15:20 ===
DEVICE: cuda
FEAT_DIR: /content/driv

  0%|          | 0/5 [00:00<?, ?it/s]
100%|██████████| 6.25M/6.25M [00:00<00:00, 229MB/s]
E0000 00:00:1771250346.958308   15763 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771250346.966526   15763 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771250346.989259   15763 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771250346.989294   15763 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771250346.989297   15763 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771250346.989299   

**FACE Variant 1: exp_face_3frames_avg**

In [ ]:
%%bash
set -euxo pipefail
CSV=/content/phase1_all_with_split.csv
BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_face_3frames_avg

mkdir -p "$EXP_DIR"
ln -sfn "$BASE_DIR"/*_ctx.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skel.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skelmask.pt "$EXP_DIR"/ 2>/dev/null || true

DEVICE=cuda
CHUNK=32
SHARD=5
export PYTHONUNBUFFERED=1

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD)); if [ $END -gt $TOTAL ]; then END=$TOTAL; fi
  echo "=== FACE 3frames_avg shard $START:$END ==="
  time env FACE_SAMPLING=3frames_avg PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 \
    python -u /content/src/cache_features_cli.py \
      --csv "$CSV" --phase 1 --feat_dir "$EXP_DIR" \
      --chunk "$CHUNK" --start "$START" --end "$END" --device "$DEVICE" \
      --do_face
  START=$END
done


=== FACE 3frames_avg shard 0:5 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_3frames_avg
MODES: ctx= False face= True skel= False
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

DONE shard [0:5) ok=5 bad=0
=== FACE 3frames_avg shard 5:10 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_3frames_avg
MODES: ctx= False face= True skel= False

DONE shard [5:10) ok=5 bad=0
=== FACE 3frames_avg shard 10:15 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_3frames_avg
MODES: ctx= False face= True skel= False

DONE shard [10:15) ok=5 bad=0
=== FACE 3frames_avg shard 15:20 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_feat

+ CSV=/content/phase1_all_with_split.csv
+ BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
+ EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_face_3frames_avg
+ mkdir -p /content/drive/MyDrive/miga_features_cache_agcn/exp_face_3frames_avg
+ ln -sfn /content/drive/MyDrive/miga_features_cache_agcn/00a3ed33d38e_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/00b40a2e4d97_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01087f1b2489_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/0123fc67d163_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/017fc7925072_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01b372587e0a_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01dc5000e304_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/03ae04661248_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/049e754be9f7_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/04c0dc00860f_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/05b

**FACE Variant 2: exp_face_centercrop_fast**

In [ ]:
%%bash
set -euxo pipefail
CSV=/content/phase1_all_with_split.csv
BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast

mkdir -p "$EXP_DIR"
ln -sfn "$BASE_DIR"/*_ctx.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skel.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_skelmask.pt "$EXP_DIR"/ 2>/dev/null || true

DEVICE=cuda
CHUNK=32
SHARD=10
export PYTHONUNBUFFERED=1

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD)); if [ $END -gt $TOTAL ]; then END=$TOTAL; fi
  echo "=== FACE center-crop shard $START:$END ==="
  time env FACE_DETECT=center PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 \
    python -u /content/src/cache_features_cli.py \
      --csv "$CSV" --phase 1 --feat_dir "$EXP_DIR" \
      --chunk "$CHUNK" --start "$START" --end "$END" --device "$DEVICE" \
      --do_face
  START=$END
done


=== FACE center-crop shard 0:10 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast
MODES: ctx= False face= True skel= False

DONE shard [0:10) ok=10 bad=0
=== FACE center-crop shard 10:20 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast
MODES: ctx= False face= True skel= False

DONE shard [10:20) ok=10 bad=0
=== FACE center-crop shard 20:30 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast
MODES: ctx= False face= True skel= False

DONE shard [20:30) ok=10 bad=0
=== FACE center-crop shard 30:40 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast
MODES: ctx= False face= True skel= False

DONE shard [30:40) ok=10 bad=0
=== FACE center-crop shard 40:50 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast
MODES: ctx= False face= True skel= False

DON

+ CSV=/content/phase1_all_with_split.csv
+ BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
+ EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast
+ mkdir -p /content/drive/MyDrive/miga_features_cache_agcn/exp_face_centercrop_fast
+ ln -sfn /content/drive/MyDrive/miga_features_cache_agcn/00a3ed33d38e_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/00b40a2e4d97_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01087f1b2489_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/0123fc67d163_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/017fc7925072_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01b372587e0a_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01dc5000e304_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/03ae04661248_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/049e754be9f7_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/04c0dc00860f_ctx.pt /content/drive/MyDrive/miga_features_cache_

**SKEL Variant 0: mean**

In [15]:
%%bash
export SKEL_AGG=mean

CHUNK=32
SHARD=10
DEVICE=cpu
CSV=/content/phase1_all_with_split.csv
BASE_FEAT_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_NAME=baseline_skel
FEAT_DIR=${BASE_FEAT_DIR}/${EXP_NAME}

mkdir -p "$FEAT_DIR"

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== SKEL baseline (mean) shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 1 --feat_dir "$FEAT_DIR" \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_skel

  START=$END
done

=== SKEL baseline (mean) shard 0:10 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_skel
MODES: ctx= False face= False skel= True

DONE shard [0:10) ok=10 bad=0
=== SKEL baseline (mean) shard 10:20 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_skel
MODES: ctx= False face= False skel= True

DONE shard [10:20) ok=10 bad=0
=== SKEL baseline (mean) shard 20:30 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_skel
MODES: ctx= False face= False skel= True

DONE shard [20:30) ok=10 bad=0
=== SKEL baseline (mean) shard 30:40 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_skel
MODES: ctx= False face= False skel= True

DONE shard [30:40) ok=10 bad=0
=== SKEL baseline (mean) shard 40:50 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/baseline_skel
MODES: ctx= False face= False skel= True

DONE shard [40:50) ok=10 bad=0
=== SKEL bas

 60%|██████    | 6/10 [00:00<00:00,  7.83it/s]Traceback (most recent call last):
  File "/content/src/cache_features_cli.py", line 46, in main
    build_and_cache_one(
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 120, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/src/feature_extract.py", line 431, in build_and_cache_one
    if spath is None or (not os.path.exists(spath)):
                             ^^^^^^^^^^^^^^^^^^^^^
  File "<frozen genericpath>", line 19, in exists
TypeError: stat: path should be string, bytes, os.PathLike or integer, not float
 20%|██        | 2/10 [00:00<00:02,  4.00it/s]Traceback (most recent call last):
  File "/content/src/cache_features_cli.py", line 46, in main
    build_and_cache_one(
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 120, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "

**SKEL Variant 1: exp_skel_mean_std**

In [ ]:
%%bash
set -euxo pipefail
CSV=/content/phase1_all_with_split.csv
BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std

mkdir -p "$EXP_DIR"
ln -sfn "$BASE_DIR"/*_ctx.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_face.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_facemask.pt "$EXP_DIR"/ 2>/dev/null || true

DEVICE=cpu
CHUNK=32
SHARD=10
export PYTHONUNBUFFERED=1

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD)); if [ $END -gt $TOTAL ]; then END=$TOTAL; fi
  echo "=== SKEL mean_std shard $START:$END ==="
  time env SKEL_AGG=mean_std PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 \
    python -u /content/src/cache_features_cli.py \
      --csv "$CSV" --phase 1 --feat_dir "$EXP_DIR" \
      --chunk "$CHUNK" --start "$START" --end "$END" --device "$DEVICE" \
      --do_skel
  START=$END
done


=== SKEL mean_std shard 0:10 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [0:10) ok=10 bad=0
=== SKEL mean_std shard 10:20 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [10:20) ok=10 bad=0
=== SKEL mean_std shard 20:30 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [20:30) ok=10 bad=0
=== SKEL mean_std shard 30:40 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [30:40) ok=10 bad=0
=== SKEL mean_std shard 40:50 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [40:50) ok=10 bad=0
=== SKEL mean_std shard 50:

+ CSV=/content/phase1_all_with_split.csv
+ BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
+ EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std
+ mkdir -p /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std
+ ln -sfn /content/drive/MyDrive/miga_features_cache_agcn/00a3ed33d38e_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/00b40a2e4d97_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01087f1b2489_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/0123fc67d163_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/017fc7925072_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01b372587e0a_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01dc5000e304_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/03ae04661248_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/049e754be9f7_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/04c0dc00860f_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/05b600b03

**SKEL Variant 2: exp_skel_max**

In [ ]:
%%bash
set -euxo pipefail
CSV=/content/phase1_all_with_split.csv
BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max

mkdir -p "$EXP_DIR"
ln -sfn "$BASE_DIR"/*_ctx.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_face.pt "$EXP_DIR"/ 2>/dev/null || true
ln -sfn "$BASE_DIR"/*_facemask.pt "$EXP_DIR"/ 2>/dev/null || true

DEVICE=cpu
CHUNK=32
SHARD=10
export PYTHONUNBUFFERED=1

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase1_all_with_split.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD)); if [ $END -gt $TOTAL ]; then END=$TOTAL; fi
  echo "=== SKEL max shard $START:$END ==="
  time env SKEL_AGG=max PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 \
    python -u /content/src/cache_features_cli.py \
      --csv "$CSV" --phase 1 --feat_dir "$EXP_DIR" \
      --chunk "$CHUNK" --start "$START" --end "$END" --device "$DEVICE" \
      --do_skel
  START=$END
done


=== SKEL max shard 0:10 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max
MODES: ctx= False face= False skel= True

DONE shard [0:10) ok=10 bad=0
=== SKEL max shard 10:20 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max
MODES: ctx= False face= False skel= True

DONE shard [10:20) ok=10 bad=0
=== SKEL max shard 20:30 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max
MODES: ctx= False face= False skel= True

DONE shard [20:30) ok=10 bad=0
=== SKEL max shard 30:40 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max
MODES: ctx= False face= False skel= True

DONE shard [30:40) ok=10 bad=0
=== SKEL max shard 40:50 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max
MODES: ctx= False face= False skel= True

DONE shard [40:50) ok=10 bad=0
=== SKEL max shard 50:60 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/mig

+ CSV=/content/phase1_all_with_split.csv
+ BASE_DIR=/content/drive/MyDrive/miga_features_cache_agcn
+ EXP_DIR=/content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max
+ mkdir -p /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_max
+ ln -sfn /content/drive/MyDrive/miga_features_cache_agcn/00a3ed33d38e_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/00b40a2e4d97_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01087f1b2489_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/0123fc67d163_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/017fc7925072_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01b372587e0a_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/01dc5000e304_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/03ae04661248_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/049e754be9f7_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/04c0dc00860f_ctx.pt /content/drive/MyDrive/miga_features_cache_agcn/05b600b0301d_ctx.pt

In [27]:
%%writefile /content/src/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.GELU(),
            nn.Linear(d // 2, 1)
        )

    def forward(self, x, mask):
        # x: (B, T, d)
        # mask: (B, T) – True для реальных данных
        if mask.numel() == 0 or mask.sum() == 0:
            # Нет валидных элементов – возвращаем нулевой вектор
            return torch.zeros(x.size(0), x.size(2), device=x.device, dtype=x.dtype)
        score = self.scorer(x).squeeze(-1)  # (B, T)
        score = score.float()
        score = score.masked_fill(~mask, -1e4)
        w = torch.softmax(score, dim=1).unsqueeze(-1)  # (B, T, 1)
        w = w.to(dtype=x.dtype)
        return (x * w).sum(dim=1)

class TriStreamModel(nn.Module):
    def __init__(
        self,
        d_ctx_in=512,
        d_face_in=1280,
        d_skel_in=512,
        d=512,
        n_layers=4,
        n_heads=4,
        dropout=0.3,
    ):
        super().__init__()

        self.ctx_in  = nn.Sequential(nn.Linear(d_ctx_in, d), nn.Dropout(dropout))
        self.face_in = nn.Sequential(nn.Linear(d_face_in, d), nn.Dropout(dropout))
        self.skel_in = nn.Sequential(nn.Linear(d_skel_in, d), nn.Dropout(dropout))

        enc = nn.TransformerEncoderLayer(
            d_model=d,
            nhead=n_heads,
            dim_feedforward=4 * d,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.ctx_enc  = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.face_enc = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.skel_enc = nn.TransformerEncoder(enc, num_layers=n_layers)

        self.ctx_pool  = AttnPool(d)
        self.face_pool = AttnPool(d)
        self.skel_pool = AttnPool(d)

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(3 * d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 1)
        )

    def forward(self, ctx, face, skel, time_mask, face_mask, skel_mask):
        # Проецируем
        x_ctx = self.ctx_in(ctx)
        x_face = self.face_in(face)
        x_skel = self.skel_in(skel)

        # Кодируем контекст (всегда есть)
        if x_ctx.size(1) > 0:
            pad_ignore = ~time_mask  # (B, T)
            # Если в каком-то образце все элементы pad_ignore True, исправляем
            if pad_ignore.ndim == 2 and pad_ignore.shape[1] > 0:
                all_pad = pad_ignore.all(dim=1)
                if all_pad.any():
                    pad_ignore = pad_ignore.clone()
                    pad_ignore[all_pad, 0] = False
            x_ctx = self.ctx_enc(x_ctx, src_key_padding_mask=pad_ignore)

        # Кодируем лицо, только если есть хотя бы один ненулевой элемент маски
        if x_face.size(1) > 0 and face_mask.sum() > 0:
            x_face = self.face_enc(x_face, src_key_padding_mask=~face_mask)
        # иначе оставляем x_face как есть (уже спроецирован, но не обработан энкодером)

        # Кодируем скелет аналогично
        if x_skel.size(1) > 0 and skel_mask.sum() > 0:
            x_skel = self.skel_enc(x_skel, src_key_padding_mask=~skel_mask)

        # Пулинг с учётом масок
        ctx_vec  = self.ctx_pool(x_ctx, time_mask)
        face_vec = self.face_pool(x_face, face_mask)
        skel_vec = self.skel_pool(x_skel, skel_mask)

        # Конкатенация и классификация
        z = torch.cat([ctx_vec, face_vec, skel_vec], dim=-1)
        return self.head(z).squeeze(-1)


Overwriting /content/src/model.py


In [15]:
%%writefile /content/src/ds.py
import os
import torch
import numpy as np
from torch.utils.data import Dataset

def cache_key(video_id, split, phase):
    s = f"{int(video_id)}|{split}|{phase}"
    import hashlib
    return hashlib.md5(s.encode("utf-8")).hexdigest()[:12]

class Track3CachedDataset(Dataset):
    def __init__(
        self,
        df,
        ctx_dir=None,
        face_dir=None,
        skel_dir=None,
        feat_dir=None,
        phase=1,
        has_label=True,
        use_ctx=True,
        use_face=True,
        use_skel=True
    ):
        self.df = df.reset_index(drop=True)
        self.phase = phase
        self.has_label = has_label
        self.use_ctx = use_ctx
        self.use_face = use_face
        self.use_skel = use_skel

        if feat_dir is not None:
            self.ctx_dir = feat_dir
            self.face_dir = feat_dir
            self.skel_dir = feat_dir
        else:
            self.ctx_dir = ctx_dir
            self.face_dir = face_dir
            self.skel_dir = skel_dir

        self.keys = []
        self.ids = []
        self.splits = []
        for _, row in self.df.iterrows():
            video_id = int(row["video_id"])
            split = row.get("split", "train")
            key = cache_key(video_id, split, self.phase)
            self.keys.append(key)
            self.ids.append(video_id)
            self.splits.append(split)

        if has_label and "label" in self.df.columns:
            self.labels = self.df["label"].values.astype(np.float32)
        else:
            self.labels = None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        key = self.keys[idx]
        out = {"id": self.ids[idx], "split": self.splits[idx]}

        # Контекст
        if self.use_ctx and self.ctx_dir:
            ctx_path = os.path.join(self.ctx_dir, f"{key}_ctx.pt")
            if os.path.exists(ctx_path):
                ctx = torch.load(ctx_path)
                out["ctx"] = ctx.float()
                out["time_mask"] = torch.ones(len(ctx), dtype=torch.bool)
            else:
                out["ctx"] = torch.zeros((0, 512), dtype=torch.float32)
                out["time_mask"] = torch.zeros(0, dtype=torch.bool)
        else:
            out["ctx"] = torch.zeros((0, 512), dtype=torch.float32)
            out["time_mask"] = torch.zeros(0, dtype=torch.bool)

        # Лицо
        if self.use_face and self.face_dir:
            face_path = os.path.join(self.face_dir, f"{key}_face.pt")
            fmask_path = os.path.join(self.face_dir, f"{key}_facemask.pt")
            if os.path.exists(face_path) and os.path.exists(fmask_path):
                face = torch.load(face_path)
                fmask = torch.load(fmask_path)
                out["face"] = face.float()
                out["face_mask"] = fmask.bool()
            else:
                out["face"] = torch.zeros((0, 1280), dtype=torch.float32)
                out["face_mask"] = torch.zeros(0, dtype=torch.bool)
        else:
            out["face"] = torch.zeros((0, 1280), dtype=torch.float32)
            out["face_mask"] = torch.zeros(0, dtype=torch.bool)

        # Скелет
        if self.use_skel and self.skel_dir:
            skel_path = os.path.join(self.skel_dir, f"{key}_skel.pt")
            smask_path = os.path.join(self.skel_dir, f"{key}_skelmask.pt")
            if os.path.exists(skel_path) and os.path.exists(smask_path):
                skel = torch.load(skel_path)
                smask = torch.load(smask_path)
                out["skel"] = skel.float()
                out["skel_mask"] = smask.bool()
            else:
                out["skel"] = torch.zeros((0, 512), dtype=torch.float32)
                out["skel_mask"] = torch.zeros(0, dtype=torch.bool)
        else:
            out["skel"] = torch.zeros((0, 512), dtype=torch.float32)
            out["skel_mask"] = torch.zeros(0, dtype=torch.bool)

        if self.labels is not None:
            out["y"] = torch.tensor(self.labels[idx], dtype=torch.float32)

        return out

def pad_seq(list_TD, pad_value=0.0):
    if len(list_TD) == 0:
        return torch.empty(0, 0, 0), torch.empty(0, 0, dtype=torch.bool)
    T = max(x.shape[0] for x in list_TD)
    D = list_TD[0].shape[1] if len(list_TD[0].shape) > 1 else 0
    out = torch.full((len(list_TD), T, D), pad_value, dtype=torch.float32)
    mask = torch.zeros((len(list_TD), T), dtype=torch.bool)
    for i, x in enumerate(list_TD):
        if x.shape[0] > 0:
            out[i, :x.shape[0]] = x.float()
            mask[i, :x.shape[0]] = True
    return out, mask

def collate_fn(batch):
    # Контекст – всегда есть (может быть пустым, но для тренировки не пуст)
    ctx_list = [b["ctx"] for b in batch]
    ctx_pad, time_mask = pad_seq(ctx_list)  # (B, T_ctx, 512)
    B, T_ctx, _ = ctx_pad.shape

    # Лицо: если хотя бы один образец имеет ненулевую длину, паддим до T_ctx,
    # иначе возвращаем пустой тензор и пустую маску.
    face_lens = [b["face"].size(0) for b in batch]
    if any(l > 0 for l in face_lens):
        face_pad = torch.zeros(B, T_ctx, 1280, dtype=torch.float32)
        face_mask = torch.zeros(B, T_ctx, dtype=torch.bool)
        for i, b in enumerate(batch):
            f = b["face"]
            if f.size(0) > 0:
                copy_len = min(T_ctx, f.size(0))
                face_pad[i, :copy_len] = f[:copy_len]
                fm = b.get("face_mask", torch.ones(copy_len, dtype=torch.bool))
                face_mask[i, :copy_len] = fm[:copy_len]
    else:
        # Все образцы не имеют лица -> возвращаем пустые тензоры
        face_pad = torch.zeros(B, 0, 1280, dtype=torch.float32)
        face_mask = torch.zeros(B, 0, dtype=torch.bool)

    # Скелет аналогично
    skel_lens = [b["skel"].size(0) for b in batch]
    if any(l > 0 for l in skel_lens):
        skel_pad = torch.zeros(B, T_ctx, 512, dtype=torch.float32)
        skel_mask = torch.zeros(B, T_ctx, dtype=torch.bool)
        for i, b in enumerate(batch):
            s = b["skel"]
            if s.size(0) > 0:
                copy_len = min(T_ctx, s.size(0))
                skel_pad[i, :copy_len] = s[:copy_len]
                sm = b.get("skel_mask", torch.ones(copy_len, dtype=torch.bool))
                skel_mask[i, :copy_len] = sm[:copy_len]
    else:
        skel_pad = torch.zeros(B, 0, 512, dtype=torch.float32)
        skel_mask = torch.zeros(B, 0, dtype=torch.bool)

    out = {
        "id": [b["id"] for b in batch],
        "ctx": ctx_pad,
        "face": face_pad,
        "skel": skel_pad,
        "time_mask": time_mask,
        "face_mask": face_mask,
        "skel_mask": skel_mask,
    }

    if "y" in batch[0]:
        out["y"] = torch.stack([b["y"] for b in batch])

    return out

Writing /content/src/ds.py


**PHASE 1**

**Абляция по модальностям**

In [16]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold

from src.ds import Track3CachedDataset, collate_fn
from src.model import TriStreamModel

# ---- базовые пути ----
BASE_FEAT_DIR = "/content/drive/MyDrive/miga_features_cache_agcn"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---- загружаем данные ----
if 'train_ok' not in dir() or train_ok is None or len(train_ok) == 0:
    train_ok = pd.read_csv('/content/phase1_all_with_split.csv')
    print(f"Загружено {len(train_ok)} строк")

df_ok = train_ok.reset_index(drop=True).copy()
print(f"Размер df_ok: {len(df_ok)}")

# ---- определяем размерности признаков на примере baseline-папок ----
tmp_ds = Track3CachedDataset(
    df_ok.iloc[:1],
    ctx_dir=os.path.join(BASE_FEAT_DIR, "baseline_ctx"),
    face_dir=os.path.join(BASE_FEAT_DIR, "baseline_face"),
    skel_dir=os.path.join(BASE_FEAT_DIR, "baseline_skel"),
    phase=1,
    has_label=True
)
sample = tmp_ds[0]

D_FACE  = int(sample["face"].shape[1])
d_ctx_in  = int(sample["ctx"].shape[1])
d_skel_in = int(sample["skel"].shape[1])

print("sample id:", sample["id"])
print("dims: ctx", d_ctx_in, "face", D_FACE, "skel", d_skel_in)

# =========================
# Metrics helpers
# =========================
@torch.no_grad()
def probs_from_logits(logits: torch.Tensor) -> np.ndarray:
    return torch.sigmoid(logits).detach().cpu().numpy().astype(np.float64)

def top1_at_thr(p: np.ndarray, y: np.ndarray, thr: float = 0.5) -> float:
    pred = (p >= thr).astype(np.int64)
    return float((pred == y.astype(np.int64)).mean())

def balanced_top1_at_thr(p: np.ndarray, y: np.ndarray, thr: float = 0.5):
    y = y.astype(np.int64)
    pred = (p >= thr).astype(np.int64)
    pos = (y == 1)
    neg = (y == 0)
    tpr = float((pred[pos] == 1).mean()) if pos.any() else float("nan")
    tnr = float((pred[neg] == 0).mean()) if neg.any() else float("nan")
    bacc = float(np.nanmean([tpr, tnr]))
    return bacc, tpr, tnr

# =========================
# Ablation helpers
# =========================
def apply_ablation_to_batch(b, cfg):
    if not cfg["use_ctx"]:
        b["ctx"] = torch.zeros_like(b["ctx"])
    if not cfg["use_face"]:
        b["face"] = torch.zeros_like(b["face"])
        b["face_mask"] = torch.zeros_like(b["face_mask"]).bool()
    if not cfg["use_skel"]:
        b["skel"] = torch.zeros_like(b["skel"])
        b["skel_mask"] = torch.zeros_like(b["skel_mask"]).bool()
    return b

@torch.no_grad()
def infer(model, loader, ab_cfg):
    model.eval()
    all_p, all_y, all_ids = [], [], []
    for b in loader:
        b = apply_ablation_to_batch(b, ab_cfg)
        logit = model(
            b["ctx"].to(device, non_blocking=True),
            b["face"].to(device, non_blocking=True),
            b["skel"].to(device, non_blocking=True),
            b["time_mask"].to(device, non_blocking=True),
            b["face_mask"].to(device, non_blocking=True),
            b["skel_mask"].to(device, non_blocking=True),
        )
        p = probs_from_logits(logit)
        y = b["y"].detach().cpu().numpy().astype(np.int64)
        ids = np.array(b["id"], dtype=np.int64)
        all_p.append(p)
        all_y.append(y)
        all_ids.append(ids)
    return np.concatenate(all_p), np.concatenate(all_y), np.concatenate(all_ids)

# =========================
# Train one fold
# =========================
def train_one_fold(
    tr_df, va_df, ab_cfg,
    ctx_dir, face_dir, skel_dir,
    epochs=12, lr=2e-4, wd=1e-4, bs=16, patience=4
):
    tr_ds = Track3CachedDataset(
        tr_df,
        ctx_dir=ctx_dir, face_dir=face_dir, skel_dir=skel_dir,
        phase=1, has_label=True,
        use_ctx=ab_cfg["use_ctx"], use_face=ab_cfg["use_face"], use_skel=ab_cfg["use_skel"]
    )
    va_ds = Track3CachedDataset(
        va_df,
        ctx_dir=ctx_dir, face_dir=face_dir, skel_dir=skel_dir,
        phase=1, has_label=True,
        use_ctx=ab_cfg["use_ctx"], use_face=ab_cfg["use_face"], use_skel=ab_cfg["use_skel"]
    )

    y_tr = tr_df["label"].values.astype(np.int64)
    cnt = np.bincount(y_tr, minlength=2)
    cls_w = 1.0 / np.maximum(cnt, 1)
    samp_w = cls_w[y_tr]
    sampler = WeightedRandomSampler(samp_w, num_samples=len(samp_w), replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=bs, sampler=sampler, num_workers=0,
                           collate_fn=collate_fn, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=bs, shuffle=False, num_workers=0,
                           collate_fn=collate_fn, pin_memory=True)

    model = TriStreamModel(
        d_ctx_in=d_ctx_in,
        d_face_in=D_FACE,
        d_skel_in=d_skel_in,
        d=512,
        n_layers=4,
        n_heads=4,
        dropout=0.3,
    ).to(device)

    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

    best_acc = -1.0
    best_state = None
    bad = 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for b in tr_loader:
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                logit = model(
                    b["ctx"].to(device, non_blocking=True),
                    b["face"].to(device, non_blocking=True),
                    b["skel"].to(device, non_blocking=True),
                    b["time_mask"].to(device, non_blocking=True),
                    b["face_mask"].to(device, non_blocking=True),
                    b["skel_mask"].to(device, non_blocking=True),
                )
                yb = b["y"].to(device, non_blocking=True)
                loss = crit(logit, yb)
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.item()))

        p, y, _ = infer(model, va_loader, ab_cfg)
        acc05 = top1_at_thr(p, y, 0.5)
        bacc05, tpr, tnr = balanced_top1_at_thr(p, y, 0.5)

        print(f"epoch {epoch}: loss={np.mean(losses):.4f} "
              f"val_top1@0.5={acc05:.4f} | bal_top1={bacc05:.4f} (tpr={tpr:.3f}, tnr={tnr:.3f})")

        if acc05 > best_acc + 1e-6:
            best_acc = acc05
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    return best_acc, best_state

# =========================
# Run CV for one setting
# =========================
def run_cv_setting(df, name, ab_cfg, ctx_dir, face_dir, skel_dir,
                   n_splits=5, seed=42, epochs=12, lr=2e-4, wd=1e-4, bs=16):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y_all = df["label"].values.astype(np.int64)

    oof_p = np.zeros(len(df), dtype=np.float64)
    fold_scores = []
    best_fold_state = None
    best_fold_score = -1.0

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, y_all), 1):
        print(f"\n--- fold {fold}/{n_splits} ---")
        tr_df = df.iloc[tr_idx].reset_index(drop=True)
        va_df = df.iloc[va_idx].reset_index(drop=True)

        fold_acc, fold_state = train_one_fold(
            tr_df, va_df, ab_cfg,
            ctx_dir=ctx_dir, face_dir=face_dir, skel_dir=skel_dir,
            epochs=epochs, lr=lr, wd=wd, bs=bs, patience=4
        )

        model = TriStreamModel(
            d_ctx_in=d_ctx_in,
            d_face_in=D_FACE,
            d_skel_in=d_skel_in,
            d=512,
            n_layers=4,
            n_heads=4,
            dropout=0.3,
        ).to(device)
        model.load_state_dict(fold_state)
        model.eval()

        va_ds = Track3CachedDataset(va_df, ctx_dir=ctx_dir, face_dir=face_dir, skel_dir=skel_dir,
                                     phase=1, has_label=True,
                                     use_ctx=ab_cfg["use_ctx"], use_face=ab_cfg["use_face"], use_skel=ab_cfg["use_skel"])
        va_loader = DataLoader(va_ds, batch_size=bs, shuffle=False, num_workers=0,
                               collate_fn=collate_fn, pin_memory=True)

        p, y, _ = infer(model, va_loader, ab_cfg)
        oof_p[va_idx] = p

        fold_scores.append(fold_acc)
        print(f"fold {fold} best top1@0.5={fold_acc:.4f}")

        if fold_acc > best_fold_score:
            best_fold_score = fold_acc
            best_fold_state = fold_state

    oof_acc05 = top1_at_thr(oof_p, y_all, 0.5)
    oof_bacc05, oof_tpr, oof_tnr = balanced_top1_at_thr(oof_p, y_all, 0.5)

    print(f"\n[{name}] CV mean top1@0.5={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"[{name}] OOF  top1@0.5={oof_acc05:.4f} | bal_top1={oof_bacc05:.4f} (tpr={oof_tpr:.3f}, tnr={oof_tnr:.3f})")

    return {
        "name": name,
        "ab_cfg": ab_cfg,
        "fold_scores": fold_scores,
        "oof_p": oof_p,
        "oof_y": y_all.copy(),
        "best_state": best_fold_state,
        "oof_acc05": oof_acc05,
        "oof_bacc05": oof_bacc05,
    }

# =========================
# ABLATIONS – только унимодальные эксперименты
# =========================
BASE = "/content/drive/MyDrive/miga_features_cache_agcn"

# Конфигурации для каждой модальности
unimodal_cfgs = {
    "ctx":  {"use_ctx": True,  "use_face": False, "use_skel": False},
    "face": {"use_ctx": False, "use_face": True,  "use_skel": False},
    "skel": {"use_ctx": False, "use_face": False, "use_skel": True},
}

# Список экспериментов с указанием модальности и путей
experiments = [
    # Контекст
    {"name": "ctx_baseline", "modality": "ctx", "ctx_dir": f"{BASE}/baseline_ctx", "face_dir": f"{BASE}/baseline_face", "skel_dir": f"{BASE}/baseline_skel"},
    {"name": "ctx_mc3",      "modality": "ctx", "ctx_dir": f"{BASE}/exp_ctx_mc3", "face_dir": f"{BASE}/baseline_face", "skel_dir": f"{BASE}/baseline_skel"},
    {"name": "ctx_r2plus1d", "modality": "ctx", "ctx_dir": f"{BASE}/exp_ctx_r2plus1d", "face_dir": f"{BASE}/baseline_face", "skel_dir": f"{BASE}/baseline_skel"},
    # Лицо
    {"name": "face_baseline", "modality": "face", "ctx_dir": f"{BASE}/baseline_ctx", "face_dir": f"{BASE}/baseline_face", "skel_dir": f"{BASE}/baseline_skel"},
    {"name": "face_3avg",     "modality": "face", "ctx_dir": f"{BASE}/baseline_ctx", "face_dir": f"{BASE}/exp_face_3frames_avg", "skel_dir": f"{BASE}/baseline_skel"},
    {"name": "face_center",   "modality": "face", "ctx_dir": f"{BASE}/baseline_ctx", "face_dir": f"{BASE}/exp_face_centercrop_fast", "skel_dir": f"{BASE}/baseline_skel"},
    # Скелет
    {"name": "skel_baseline", "modality": "skel", "ctx_dir": f"{BASE}/baseline_ctx", "face_dir": f"{BASE}/baseline_face", "skel_dir": f"{BASE}/baseline_skel"},
    {"name": "skel_meanstd",  "modality": "skel", "ctx_dir": f"{BASE}/baseline_ctx", "face_dir": f"{BASE}/baseline_face", "skel_dir": f"{BASE}/exp_skel_mean_std"},
    {"name": "skel_max",      "modality": "skel", "ctx_dir": f"{BASE}/baseline_ctx", "face_dir": f"{BASE}/baseline_face", "skel_dir": f"{BASE}/exp_skel_max"},
]

summary_rows = []

for exp in experiments:
    modality = exp["modality"]
    ab_cfg = unimodal_cfgs[modality]
    print("\n" + "="*80)
    print(f"Эксперимент: {exp['name']} | Модальность: {modality}")
    print(f"ctx_dir: {exp['ctx_dir']}")
    print(f"face_dir: {exp['face_dir']}")
    print(f"skel_dir: {exp['skel_dir']}")
    print("="*80)

    res = run_cv_setting(
        df_ok,
        name=exp['name'],
        ab_cfg=ab_cfg,
        ctx_dir=exp['ctx_dir'],
        face_dir=exp['face_dir'],
        skel_dir=exp['skel_dir'],
        n_splits=5,
        epochs=12
    )

    variant = exp['name'].split('_', 1)[1] if '_' in exp['name'] else exp['name']

    summary_rows.append({
        "Experiment": exp['name'],
        "Modality": modality,
        "Variant": variant,
        "Top-1 Accuracy (%)": 100.0 * res["oof_acc05"],
        "CV mean (%)": 100.0 * np.mean(res["fold_scores"]),
        "CV std (%)": 100.0 * np.std(res["fold_scores"]),
    })

summary = pd.DataFrame(summary_rows)
summary = summary.sort_values(["Modality", "Top-1 Accuracy (%)"], ascending=[True, False])
display(summary)


device: cuda
Загружено 255 строк
Размер df_ok: 255
sample id: 68
dims: ctx 512 face 1280 skel 512

Эксперимент: ctx_baseline | Модальность: ctx
ctx_dir: /content/drive/MyDrive/miga_features_cache_agcn/baseline_ctx
face_dir: /content/drive/MyDrive/miga_features_cache_agcn/baseline_face
skel_dir: /content/drive/MyDrive/miga_features_cache_agcn/baseline_skel

--- fold 1/5 ---


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


epoch 1: loss=0.7282 val_top1@0.5=0.8039 | bal_top1=0.5000 (tpr=1.000, tnr=0.000)
epoch 2: loss=0.7322 val_top1@0.5=0.1961 | bal_top1=0.5000 (tpr=0.000, tnr=1.000)
epoch 3: loss=0.6747 val_top1@0.5=0.8039 | bal_top1=0.5000 (tpr=1.000, tnr=0.000)
epoch 4: loss=0.7035 val_top1@0.5=0.3922 | bal_top1=0.5463 (tpr=0.293, tnr=0.800)
epoch 5: loss=0.6824 val_top1@0.5=0.7843 | bal_top1=0.4878 (tpr=0.976, tnr=0.000)
fold 1 best top1@0.5=0.8039

--- fold 2/5 ---
epoch 1: loss=0.6926 val_top1@0.5=0.1961 | bal_top1=0.5000 (tpr=0.000, tnr=1.000)
epoch 2: loss=0.6999 val_top1@0.5=0.3922 | bal_top1=0.3573 (tpr=0.415, tnr=0.300)
epoch 3: loss=0.6695 val_top1@0.5=0.1961 | bal_top1=0.5000 (tpr=0.000, tnr=1.000)
epoch 4: loss=0.7065 val_top1@0.5=0.6863 | bal_top1=0.4646 (tpr=0.829, tnr=0.100)
epoch 5: loss=0.6562 val_top1@0.5=0.7255 | bal_top1=0.4890 (tpr=0.878, tnr=0.100)
epoch 6: loss=0.5836 val_top1@0.5=0.6275 | bal_top1=0.4659 (tpr=0.732, tnr=0.200)
epoch 7: loss=0.5846 val_top1@0.5=0.7451 | bal_top1=

,Experiment,Modality,Variant,Top-1 Accuracy (%),CV mean (%),CV std (%)
2,ctx_r2plus1d,ctx,r2plus1d,79.607843,79.607843,0.960584
0,ctx_baseline,ctx,baseline,79.215686,79.215686,0.960584
1,ctx_mc3,ctx,mc3,77.647059,77.647059,5.628510
3,face_baseline,face,baseline,44.313725,44.313725,28.673077
5,face_center,face,center,44.313725,44.313725,28.673077
4,face_3avg,face,3avg,43.529412,43.529412,28.506324
7,skel_meanstd,skel,meanstd,56.470588,56.470588,28.506324
6,skel_baseline,skel,baseline,55.686275,55.686275,28.673077
8,skel_max,skel,max,44.313725,44.313725,28.673077


**Для полной модели выбираем:**

ctx: exp_ctx_r2plus1d

face: baseline_face

skel: exp_skel_mean_std

In [53]:
import numpy as np
import pandas as pd
import os
import torch
from sklearn.model_selection import StratifiedKFold

BASE = "/content/drive/MyDrive/miga_features_cache_agcn"

best_ctx_dir = f"{BASE}/exp_ctx_r2plus1d"
best_face_dir = f"{BASE}/baseline_face"
best_skel_dir = f"{BASE}/exp_skel_mean_std"

# Настройка для полной модели (все модальности включены)
ab_cfg_all = {"use_ctx": True, "use_face": True, "use_skel": True}

print("Запуск полной трёхмодальной модели")
print(f"ctx_dir: {best_ctx_dir}")
print(f"face_dir: {best_face_dir}")
print(f"skel_dir: {best_skel_dir}")


# Запуск CV
result = run_cv_setting(
    df=df_ok,
    name="full_model_best",
    ab_cfg=ab_cfg_all,
    ctx_dir=best_ctx_dir,
    face_dir=best_face_dir,
    skel_dir=best_skel_dir,
    n_splits=5,
    epochs=12,
    lr=2e-4,
    wd=1e-4,
    bs=16
)

# Вывод итоговых метрик
print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ ПОЛНОЙ МОДЕЛИ")
print("="*60)
print(f"OOF Top-1 Accuracy: {result['oof_acc05']*100:.2f}%")
print(f"OOF Balanced Accuracy: {result['oof_bacc05']*100:.2f}%")
print(f"CV mean accuracy: {np.mean(result['fold_scores'])*100:.2f}% ± {np.std(result['fold_scores'])*100:.2f}%")
print("="*60)

# Сохраняем результаты для дальнейшего использования
import pickle

# Сохраняем OOF-предсказания и метки
oof_data = {
    'y_true': result['oof_y'],
    'y_pred_proba': result['oof_p'],
    'fold_scores': result['fold_scores'],
    'best_state': result['best_state']
}

with open('/content/full_model_results.pkl', 'wb') as f:
    pickle.dump(oof_data, f)

print("\n Результаты сохранены в файл 'full_model_results.pkl'")

Запуск полной трёхмодальной модели
ctx_dir: /content/drive/MyDrive/miga_features_cache_agcn/exp_ctx_r2plus1d
face_dir: /content/drive/MyDrive/miga_features_cache_agcn/baseline_face
skel_dir: /content/drive/MyDrive/miga_features_cache_agcn/exp_skel_mean_std

--- fold 1/5 ---
epoch 1: loss=0.6920 val_top1@0.5=0.4706 | bal_top1=0.5573 (tpr=0.415, tnr=0.700)
epoch 2: loss=0.6675 val_top1@0.5=0.3333 | bal_top1=0.5854 (tpr=0.171, tnr=1.000)
epoch 3: loss=0.6648 val_top1@0.5=0.4706 | bal_top1=0.5951 (tpr=0.390, tnr=0.800)
epoch 4: loss=0.6320 val_top1@0.5=0.7059 | bal_top1=0.5524 (tpr=0.805, tnr=0.300)
epoch 5: loss=0.7480 val_top1@0.5=0.5686 | bal_top1=0.5805 (tpr=0.561, tnr=0.600)
epoch 6: loss=0.6443 val_top1@0.5=0.6863 | bal_top1=0.5780 (tpr=0.756, tnr=0.400)
epoch 7: loss=0.6762 val_top1@0.5=0.8039 | bal_top1=0.5378 (tpr=0.976, tnr=0.100)
epoch 8: loss=0.5066 val_top1@0.5=0.7255 | bal_top1=0.5268 (tpr=0.854, tnr=0.200)
epoch 9: loss=0.6580 val_top1@0.5=0.3922 | bal_top1=0.5841 (tpr=0.268

**PHASE 2**

In [20]:
ROOT = "/content/miga_data"
FEAT_DIR_P1 = "/content/drive/MyDrive/miga_features_cache_agcn"
FEAT_DIR_P2 = "/content/drive/MyDrive/miga_features_cache_agcn_p2"


In [21]:
import os, glob
import pandas as pd

RGB_PHASE2 = f"{ROOT}/imigue_rgb_phase2"
SK_P2_TEST = f"{ROOT}/imigue_data_phase2/imigue_skeleton_test"

def vid4(x):
    return f"{int(x):04d}"

def resolve_video_path_phase2(video_id):
    v = vid4(video_id)
    p = os.path.join(RGB_PHASE2, v, f"{v}.mp4")
    if os.path.exists(p):
        return p
    hits = glob.glob(os.path.join(RGB_PHASE2, "**", f"{v}.mp4"), recursive=True)
    return hits[0] if hits else None

def resolve_skeleton_path_phase2(video_id, prefer_hand=True):
    v = vid4(video_id)
    p_hand  = os.path.join(SK_P2_TEST, v, f"{v}_light_hand.csv")
    p_light = os.path.join(SK_P2_TEST, v, f"{v}_light.csv")
    if prefer_hand and os.path.exists(p_hand):
        return p_hand
    if os.path.exists(p_light):
        return p_light
    if os.path.exists(p_hand):
        return p_hand
    return None

ids = sorted([int(os.path.basename(p)) for p in glob.glob(os.path.join(SK_P2_TEST, "[0-9][0-9][0-9][0-9]"))])

rows = []
miss_v, miss_s = 0, 0
for vid in ids:
    vpath = resolve_video_path_phase2(vid)
    spath = resolve_skeleton_path_phase2(vid)
    if vpath is None:
        miss_v += 1
    if spath is None:
        miss_s += 1
    rows.append({
        "video_id": vid,
        "split": "test",
        "video_path": vpath,
        "skeleton_path": spath
    })

phase2_df = pd.DataFrame(rows).sort_values("video_id").reset_index(drop=True)
print("Phase2 rows:", len(phase2_df), "missing video:", miss_v, "missing skeleton:", miss_s)

PHASE2_CSV = "/content/phase2_all_with_paths.csv"
phase2_df.to_csv(PHASE2_CSV, index=False)
print("Saved:", PHASE2_CSV)
phase2_df.head()

Phase2 rows: 104 missing video: 0 missing skeleton: 0
Saved: /content/phase2_all_with_paths.csv


,video_id,split,video_path,skeleton_path
0,54,test,/content/miga_data/imigue_rgb_phase2/0054/0054...,/content/miga_data/imigue_data_phase2/imigue_s...
1,67,test,/content/miga_data/imigue_rgb_phase2/0067/0067...,/content/miga_data/imigue_data_phase2/imigue_s...
2,69,test,/content/miga_data/imigue_rgb_phase2/0069/0069...,/content/miga_data/imigue_data_phase2/imigue_s...
3,70,test,/content/miga_data/imigue_rgb_phase2/0070/0070...,/content/miga_data/imigue_data_phase2/imigue_s...
4,71,test,/content/miga_data/imigue_rgb_phase2/0071/0071...,/content/miga_data/imigue_data_phase2/imigue_s...


**Контекст (r2plus1d_18)**

In [22]:
%%bash
export CTX_BACKBONE=r2plus1d_18

CHUNK=64
SHARD=10
DEVICE=cuda
CSV=/content/phase2_all_with_paths.csv
BASE_FEAT_DIR_P2=/content/drive/MyDrive/miga_features_cache_agcn_p2
EXP_NAME=exp_ctx_r2plus1d
FEAT_DIR=${BASE_FEAT_DIR_P2}/${EXP_NAME}

mkdir -p "$FEAT_DIR"

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase2_all_with_paths.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== CTX phase2 (r2plus1d) shard $START:$END (chunk=$CHUNK) ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 2 --feat_dir "$FEAT_DIR" \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_ctx

  START=$END
done

=== CTX phase2 (r2plus1d) shard 0:10 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False
Downloading: "https://download.pytorch.org/models/r2plus1d_18-91a641e6.pth" to /root/.cache/torch/hub/checkpoints/r2plus1d_18-91a641e6.pth

DONE shard [0:10) ok=10 bad=0
=== CTX phase2 (r2plus1d) shard 10:20 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False

DONE shard [10:20) ok=10 bad=0
=== CTX phase2 (r2plus1d) shard 20:30 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False

DONE shard [20:30) ok=10 bad=0
=== CTX phase2 (r2plus1d) shard 30:40 (chunk=64) ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_ctx_r2plus1d
MODES: ctx= True face= False skel= False

DONE shard [30:40) ok=10 b

  0%|          | 0/10 [00:00<?, ?it/s]
  0%|          | 0.00/120M [00:00<?, ?B/s]
 16%|█▋        | 19.6M/120M [00:00<00:00, 205MB/s]
 34%|███▍      | 40.9M/120M [00:00<00:00, 215MB/s]
 51%|█████▏    | 61.9M/120M [00:00<00:00, 217MB/s]
 69%|██████▉   | 83.1M/120M [00:00<00:00, 219MB/s]
100%|██████████| 120M/120M [00:00<00:00, 219MB/s]
100%|██████████| 4/4 [03:23<00:00, 50.76s/it]


**Лицо (baseline: mid + yolo)**

In [23]:
%%bash
export FACE_SAMPLING=mid
export FACE_DETECT=yolo

CHUNK=32
SHARD=5
DEVICE=cuda
CSV=/content/phase2_all_with_paths.csv
BASE_FEAT_DIR_P2=/content/drive/MyDrive/miga_features_cache_agcn_p2
EXP_NAME=baseline_face
FEAT_DIR=${BASE_FEAT_DIR_P2}/${EXP_NAME}

mkdir -p "$FEAT_DIR"

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase2_all_with_paths.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== FACE phase2 (baseline) shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 2 --feat_dir "$FEAT_DIR" \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_face

  START=$END
done

=== FACE phase2 (baseline) shard 0:5 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/baseline_face
MODES: ctx= False face= True skel= False
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

DONE shard [0:5) ok=5 bad=0
=== FACE phase2 (baseline) shard 5:10 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/baseline_face
MODES: ctx= False face= True skel= False

DONE shard [5:10) ok=5 bad=0
=== FACE phase2 (baseline) shard 10:15 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/baseline_face
MODES: ctx= False face= True skel= False

DONE shard [10:15) ok=5 bad=0
=== FACE phase2 (baseline) shard 15:20 ===
DEVICE: cuda
FEAT_DIR: /content/drive/MyDri

  0%|          | 0/5 [00:00<?, ?it/s]
100%|██████████| 6.25M/6.25M [00:00<00:00, 146MB/s]
E0000 00:00:1771323246.758227   63198 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771323246.766003   63198 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771323246.787413   63198 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771323246.787447   63198 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771323246.787449   63198 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771323246.787451   

**Скелет (mean_std)**

In [24]:
%%bash
export SKEL_AGG=mean_std

CHUNK=32
SHARD=10
DEVICE=cpu
CSV=/content/phase2_all_with_paths.csv
BASE_FEAT_DIR_P2=/content/drive/MyDrive/miga_features_cache_agcn_p2
EXP_NAME=exp_skel_mean_std
FEAT_DIR=${BASE_FEAT_DIR_P2}/${EXP_NAME}

mkdir -p "$FEAT_DIR"

TOTAL=$(python - <<'PY'
import pandas as pd
print(len(pd.read_csv("/content/phase2_all_with_paths.csv")))
PY
)

START=0
while [ $START -lt $TOTAL ]; do
  END=$((START+SHARD))
  if [ $END -gt $TOTAL ]; then END=$TOTAL; fi

  echo "=== SKEL phase2 (mean_std) shard $START:$END ==="
  PYTHONPATH=/content PYTHONDONTWRITEBYTECODE=1 python -u /content/src/cache_features_cli.py \
    --csv $CSV --phase 2 --feat_dir "$FEAT_DIR" \
    --chunk $CHUNK --start $START --end $END --device $DEVICE \
    --do_skel

  START=$END
done

=== SKEL phase2 (mean_std) shard 0:10 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [0:10) ok=10 bad=0
=== SKEL phase2 (mean_std) shard 10:20 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [10:20) ok=10 bad=0
=== SKEL phase2 (mean_std) shard 20:30 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [20:30) ok=10 bad=0
=== SKEL phase2 (mean_std) shard 30:40 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_skel_mean_std
MODES: ctx= False face= False skel= True

DONE shard [30:40) ok=10 bad=0
=== SKEL phase2 (mean_std) shard 40:50 ===
DEVICE: cpu
FEAT_DIR: /content/drive/MyDrive/miga_features_cache_agcn_p2/exp_skel_mean_std
MODES: ctx= False face= False skel= True

100%|██████████| 4/4 [00:00<00:00,  5.29it/s]


**Inference**

In [56]:
import os
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from src.ds import Track3CachedDataset, collate_fn
from src.model import TriStreamModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

BASE_P2 = "/content/drive/MyDrive/miga_features_cache_agcn_p2"
test_ctx_dir = f"{BASE_P2}/exp_ctx_r2plus1d"
test_face_dir = f"{BASE_P2}/baseline_face"
test_skel_dir = f"{BASE_P2}/exp_skel_mean_std"

model_state_path = "/content/full_model_results.pkl"

with open(model_state_path, 'rb') as f:
    model_data = pickle.load(f)
best_state = model_data['best_state']
print("Модель загружена. Ключи:", best_state.keys() if best_state else None)


test_csv = "/content/phase2_all_with_paths.csv"
test_df = pd.read_csv(test_csv)
print(f"Загружено тестовых образцов: {len(test_df)}")


test_ds = Track3CachedDataset(
    test_df,
    ctx_dir=test_ctx_dir,
    face_dir=test_face_dir,
    skel_dir=test_skel_dir,
    phase=2,
    has_label=False,
    use_ctx=True,
    use_face=True,
    use_skel=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
    pin_memory=True
)

# ---- создаём модель и загружаем веса ----
model = TriStreamModel(
    d_ctx_in=d_ctx_in,
    d_face_in=D_FACE,
    d_skel_in=d_skel_in,
    d=512,
    n_layers=4,
    n_heads=4,
    dropout=0.3,
).to(device)

model.load_state_dict(best_state)
model.eval()
print("Модель инициализирована и веса загружены.")

# ---- инференс ----
all_ids = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        ids = batch["id"]
        logits = model(
            batch["ctx"].to(device, non_blocking=True),
            batch["face"].to(device, non_blocking=True),
            batch["skel"].to(device, non_blocking=True),
            batch["time_mask"].to(device, non_blocking=True),
            batch["face_mask"].to(device, non_blocking=True),
            batch["skel_mask"].to(device, non_blocking=True),
        )
        probs = torch.sigmoid(logits).cpu().numpy()
        all_ids.extend(ids)
        all_probs.extend(probs)

results_df = pd.DataFrame({
    "video_id": all_ids,
    "prediction": all_probs
})


output_path = "/content/phase2_predictions.csv"
results_df.to_csv(output_path, index=False)
print(f"Предсказания сохранены в {output_path}")
print(results_df.head())

device: cuda
Модель загружена. Ключи: dict_keys(['ctx_in.0.weight', 'ctx_in.0.bias', 'face_in.0.weight', 'face_in.0.bias', 'skel_in.0.weight', 'skel_in.0.bias', 'ctx_enc.layers.0.self_attn.in_proj_weight', 'ctx_enc.layers.0.self_attn.in_proj_bias', 'ctx_enc.layers.0.self_attn.out_proj.weight', 'ctx_enc.layers.0.self_attn.out_proj.bias', 'ctx_enc.layers.0.linear1.weight', 'ctx_enc.layers.0.linear1.bias', 'ctx_enc.layers.0.linear2.weight', 'ctx_enc.layers.0.linear2.bias', 'ctx_enc.layers.0.norm1.weight', 'ctx_enc.layers.0.norm1.bias', 'ctx_enc.layers.0.norm2.weight', 'ctx_enc.layers.0.norm2.bias', 'ctx_enc.layers.1.self_attn.in_proj_weight', 'ctx_enc.layers.1.self_attn.in_proj_bias', 'ctx_enc.layers.1.self_attn.out_proj.weight', 'ctx_enc.layers.1.self_attn.out_proj.bias', 'ctx_enc.layers.1.linear1.weight', 'ctx_enc.layers.1.linear1.bias', 'ctx_enc.layers.1.linear2.weight', 'ctx_enc.layers.1.linear2.bias', 'ctx_enc.layers.1.norm1.weight', 'ctx_enc.layers.1.norm1.bias', 'ctx_enc.layers.1.n